In [11]:
# First cell - Import and setup
import time
from pynq import Overlay
from pynq.lib import MicroblazeLibrary
import ipywidgets as widgets
from IPython.display import display

class I2CController:
    def __init__(self, sda_pin=2, scl_pin=3):
        self.PMOD_SDA_PIN = sda_pin
        self.PMOD_SCL_PIN = scl_pin
        
        # Initialize PYNQ overlay and I2C device
        self.overlay = Overlay('base.bit')
        self.iop = self.overlay.iop_pmoda
        self.lib = MicroblazeLibrary(self.iop, ['i2c'])
        self.i2c_device = self.lib.i2c_open(self.PMOD_SDA_PIN, self.PMOD_SCL_PIN)
        
    def write_data(self, device_addr, register_addr, data):
        """Write data to I2C device"""
        buffer = bytearray(3)
        buffer[0] = register_addr  # Register address
        buffer[1] = (data >> 8) & 0xFF  # Data MSB
        buffer[2] = data & 0xFF  # Data LSB
        
        self.i2c_device.write(device_addr, buffer, len(buffer))
        time.sleep(10e-3)  # Delay for stability
        
    def read_data(self, device_addr, num_bytes=2):
        """Read data from I2C device"""
        buffer = bytearray(num_bytes)
        self.i2c_device.read(device_addr, buffer, num_bytes)
        if num_bytes == 2:
            return (buffer[0] << 8) | buffer[1]
        return buffer
    
    def close(self):
        """Close I2C device"""
        self.i2c_device.close()

# Create I2C controller instance
i2c = I2CController()

# Create widgets for user input
device_addr_input = widgets.Text(
    value='0x77',
    description='Device Address:',
    style={'description_width': 'initial'}
)

register_addr_input = widgets.Text(
    value='0x14',
    description='Register Address:',
    style={'description_width': 'initial'}
)

data_input = widgets.Text(
    value='0xABCD',
    description='Data to Write:',
    style={'description_width': 'initial'}
)

def write_button_clicked(b):
    try:
        device_addr = int(device_addr_input.value, 16)
        register_addr = int(register_addr_input.value, 16)
        data = int(data_input.value, 16)
        
        i2c.write_data(device_addr, register_addr, data)
        print(f"Written to device {hex(device_addr)}, register {hex(register_addr)}: {hex(data)}")
        
        # Read back the data
        read_value = i2c.read_data(device_addr)
        print(f"Read back value: {hex(read_value)}")
    except ValueError as e:
        print("Error: Please enter valid hexadecimal values")
    except Exception as e:
        print(f"Error: {str(e)}")

write_button = widgets.Button(
    description='Write Data',
    style={'description_width': 'initial'}
)
write_button.on_click(write_button_clicked)

# Display the widgets
display(device_addr_input)
display(register_addr_input)
display(data_input)
display(write_button)

Text(value='0x77', description='Device Address:', style=DescriptionStyle(description_width='initial'))

Text(value='0x14', description='Register Address:', style=DescriptionStyle(description_width='initial'))

Text(value='0xABCD', description='Data to Write:', style=DescriptionStyle(description_width='initial'))

Button(description='Write Data', style=ButtonStyle())

Written to device 0x77, register 0x14: 0xabcd
Read back value: 0x0


Read value: 0xcc
